In [1]:
import os

# get Groq api key by reading the first line of the text file groq.txt

with open('groq.txt') as file:
    api_key = str(file.readline())

os.environ['GROQ_API_KEY'] = api_key

In [4]:
# Load CSV document
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(r'fake_startup_founders_europe.csv', encoding='latin-1')

In [5]:
doc = loader.load()
print(f"You have {len(doc)} documents")
print(f"You have {len(doc[0].page_content)} characters in the first document")

You have 10 documents
You have 393 characters in the first document


In [6]:
# create embeddings for all the documents

from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipykernel_4812/408979860.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2212.94it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# store embeddings in vectordatabase

from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(doc, embeddings)

In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
)

In [9]:
# setting up the retrieval function using modern LCEL approach

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Create a simple RAG chain
template = """Answer the question based only on the following context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = vectorstore.as_retriever()

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [10]:
# execute the chain

query = "I want to start a company in the financial sector. Who could be a relevant co-founder?"

result = chain.invoke(query)

In [11]:
print(result)

Based on the provided context, Liam Svensson could be a relevant co-founder for a company in the financial sector. He has experience as a Financial Analyst and Investment Advisor, and is currently working on a fintech startup, which suggests he has knowledge and expertise in the financial sector. Additionally, his educational background in finance (MSc in Finance and BSc in Economics) further supports his relevance as a potential co-founder for a financial sector company.
